# RPKM for Damage-seq around closed-region peak centers

Calculate real and simulated Damage-seq RPKM in 50-bp windows around H3K9me3 and H3K27me3 peak centers.

In [1]:
from pathlib import Path

BASE_DIR = Path('/cta/users/guneyn23')
OVERLAP_BASE_DIR = BASE_DIR / 'peak_center_20kb/noUV_closed_alltime_damage_overlaps'
REAL_OVERLAP_DIR = OVERLAP_BASE_DIR / 'real'
SIM_OVERLAP_DIR = OVERLAP_BASE_DIR / 'simulated'
REAL_DAMAGE_DIR = BASE_DIR / 'damageseq_data'
SIM_DAMAGE_DIR = REAL_DAMAGE_DIR / 'simulation'
OUTPUT_DIR = BASE_DIR / 'peak_center_20kb/rpkm/noUV_closed_damage_all'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = ['H3K9me3', 'H3K27me3']

SAMPLES = [
    'R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS',
    'R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS',
    'R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS',
    'R3Hela_1hCPD_CTTGTA_S4_hg38_primary_assembly_DS',
    'R3Hela_30m64_TGACCA_S7_hg38_primary_assembly_DS',
    'R3Hela_30mCPD_GGCTAC_S8_hg38_primary_assembly_DS',
    'R3Hela_4h64_GCCAAT_S9_hg38_primary_assembly_DS',
    'R3Hela_4hCPD_AGTCAA_S10_hg38_primary_assembly_DS',
    'R3Hela_8h64_CAGATC_S11_hg38_primary_assembly_DS',
    'R3Hela_8hCPD_AGTTCC_S12_hg38_primary_assembly_DS',
]

print(f'{len(SAMPLES)} samples x {len(REGIONS)} closed regions')

10 samples x 2 closed regions


In [2]:
def count_reads(damage_file):
    total_reads = 0

    with open(damage_file) as infile:
        for line in infile:
            if line.strip():
                total_reads += 1

    return total_reads


def calculate_rpkm(input_file, output_file, total_reads):
    with open(input_file) as infile, open(output_file, "w") as outfile:
        for line in infile:
            if not line.strip():
                continue

            columns = line.rstrip("\n").split("\t")

            start = int(columns[1])
            end = int(columns[2])
            damage_count = int(columns[-1])
            window_length = end - start

            rpkm = (
                damage_count * 1_000_000_000
                / (total_reads * window_length)
            )

            outfile.write(
                line.rstrip("\n") + "\t" + f"{rpkm:.2f}" + "\n"
            )

In [3]:
for region in REGIONS:
    region_output_dir = OUTPUT_DIR / region
    region_output_dir.mkdir(parents=True, exist_ok=True)

    for sample in SAMPLES:
        real_overlap_file = REAL_OVERLAP_DIR / f'{sample}_{region}_400windows_overlap.bed'
        sim_overlap_file = SIM_OVERLAP_DIR / f'{sample}_sim_{region}_400windows_overlap.bed'
        real_damage_file = REAL_DAMAGE_DIR / f'{sample}.bed'
        sim_damage_file = SIM_DAMAGE_DIR / f'{sample}_sim.bed'
        real_rpkm_file = region_output_dir / f'{sample}_{region}_400windows_rpkm.bed'
        sim_rpkm_file = region_output_dir / f'{sample}_sim_{region}_400windows_rpkm.bed'

        
        real_total_reads = count_reads(real_damage_file)
        sim_total_reads = count_reads(sim_damage_file)
        calculate_rpkm(real_overlap_file, real_rpkm_file, real_total_reads)
        calculate_rpkm(sim_overlap_file, sim_rpkm_file, sim_total_reads)

        print(f'Written: {real_rpkm_file}')
        print(f'Written: {sim_rpkm_file}')

Written: /cta/users/guneyn23/peak_center_20kb/rpkm/noUV_closed_damage_all/H3K9me3/R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS_H3K9me3_400windows_rpkm.bed
Written: /cta/users/guneyn23/peak_center_20kb/rpkm/noUV_closed_damage_all/H3K9me3/R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS_sim_H3K9me3_400windows_rpkm.bed
Written: /cta/users/guneyn23/peak_center_20kb/rpkm/noUV_closed_damage_all/H3K9me3/R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS_H3K9me3_400windows_rpkm.bed
Written: /cta/users/guneyn23/peak_center_20kb/rpkm/noUV_closed_damage_all/H3K9me3/R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS_sim_H3K9me3_400windows_rpkm.bed
Written: /cta/users/guneyn23/peak_center_20kb/rpkm/noUV_closed_damage_all/H3K9me3/R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS_H3K9me3_400windows_rpkm.bed
Written: /cta/users/guneyn23/peak_center_20kb/rpkm/noUV_closed_damage_all/H3K9me3/R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS_sim_H3K9me3_400windows_rpkm.bed
Written: /cta/users/guneyn23/peak_center

In [4]:
for region in REGIONS:
    for sample in SAMPLES:
        for suffix in ['', '_sim']:
            rpkm_file = OUTPUT_DIR / region / f'{sample}{suffix}_{region}_400windows_rpkm.bed'
            if rpkm_file.is_file():
                with rpkm_file.open() as infile:
                    first_row = infile.readline().rstrip()
                print(f'{rpkm_file.name}: {rpkm_file.stat().st_size / 1e9:.2f} GB')
                print(first_row)

R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	100547176	peak_1_1	0	0.00
R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS_sim_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	100547176	peak_1_1	0	0.00
R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	100547176	peak_1_1	1	0.63
R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS_sim_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	100547176	peak_1_1	1	0.63
R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	100547176	peak_1_1	1	0.81
R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS_sim_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	100547176	peak_1_1	0	0.00
R3Hela_1hCPD_CTTGTA_S4_hg38_primary_assembly_DS_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	100547176	peak_1_1	3	1.76
R3Hela_1hCPD_CTTGTA_S4_hg38_primary_assembly_DS_sim_H3K9me3_400windows_rpkm.bed: 0.28 GB
chr1	100547126	10